# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# CRC-VAL-HE-7K external-validation preflight

This notebook freezes the dataset inventory, label protocols, metadata status,
and prediction-independent attribution cohorts before any external model result is
examined.

Two estimands are kept separate:

1. **Primary:** native nine-class replication trained on NCT-CRC-HE-100K and
   evaluated once on CRC-VAL-HE-7K.
2. **Secondary:** Kather-to-CRC common-seven coarse-label transfer.

The notebook deliberately performs no model inference. Complete this preflight,
resolve the patient-manifest status, and preserve its outputs before running the
external classification and attribution notebook.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display


def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CRC_DIR = PROJECT_ROOT / 'CRC-VAL-HE-7K'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'crc_val_external' / 'preflight'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = (
    PROJECT_ROOT / 'Methods' / 'KatherRevision' / 'CRC_VAL_STUDY_CONFIG.json'
)
PATIENT_MANIFEST_PATH = Path(
    os.environ.get(
        'CRC_VAL_PATIENT_MANIFEST',
        PROJECT_ROOT / 'metadata' / 'crc_val_tile_patient_manifest.csv',
    )
)
COMPUTE_FILE_HASHES = True

config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
print('Project:', PROJECT_ROOT)
print('CRC-VAL:', CRC_DIR)
print('Outputs:', OUTPUT_DIR)
print('Patient manifest candidate:', PATIENT_MANIFEST_PATH)


## Dataset inventory

The inventory excludes non-TIFF files, validates every image header, and records a
SHA-256 hash. Hashing 7,180 files may take a few minutes but prevents silent dataset
changes between the preflight and final run.


In [ ]:
expected_classes = set(config['primary_native_replication']['classes'])
class_directories = {path.name for path in CRC_DIR.iterdir() if path.is_dir()}
assert class_directories == expected_classes, (class_directories, expected_classes)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


rows = []
for class_code in sorted(expected_classes):
    for path in sorted((CRC_DIR / class_code).glob('*.tif')):
        with Image.open(path) as image:
            width, height = image.size
            mode = image.mode
        rows.append(
            {
                'image_id': path.stem,
                'filename': path.name,
                'relative_path': str(path.relative_to(CRC_DIR)),
                'image_path': str(path.resolve()),
                'crc_class': class_code,
                'width': width,
                'height': height,
                'mode': mode,
                'sha256': sha256_file(path) if COMPUTE_FILE_HASHES else None,
            }
        )

inventory = pd.DataFrame(rows).sort_values(['crc_class', 'filename']).reset_index(drop=True)
assert len(inventory) == config['external_dataset']['expected_images']
assert inventory['filename'].is_unique
assert inventory[['width', 'height']].drop_duplicates().values.tolist() == [[224, 224]]
assert set(inventory['mode']) == {'RGB'}

inventory_path = OUTPUT_DIR / 'crc_val_inventory.csv'
inventory.to_csv(inventory_path, index=False)
stable_columns = [
    'image_id', 'filename', 'relative_path', 'crc_class',
    'width', 'height', 'mode', 'sha256',
]
stable_inventory_path = OUTPUT_DIR / 'crc_val_inventory_registry.csv'
inventory[stable_columns].to_csv(stable_inventory_path, index=False)
inventory_hash = hashlib.sha256(stable_inventory_path.read_bytes()).hexdigest()
class_counts = inventory.groupby('crc_class').size().rename('image_count').reset_index()
class_counts.to_csv(OUTPUT_DIR / 'crc_val_class_counts.csv', index=False)
display(class_counts)
print('Inventory SHA-256:', inventory_hash)


## Patient/source metadata gate

The distributed filenames do not identify patients or slides. A confirmatory
patient-aware analysis requires an official manifest with `filename` and
`patient_id`; `slide_id` is optional. The join below is strict and never guesses
groups from filename fragments.


In [ ]:
patient_metadata_available = PATIENT_MANIFEST_PATH.is_file()
if patient_metadata_available:
    source = pd.read_csv(PATIENT_MANIFEST_PATH)
    required = {'filename', 'patient_id'}
    missing = required.difference(source.columns)
    if missing:
        raise ValueError(f'Patient manifest is missing columns: {sorted(missing)}')
    if source['filename'].duplicated().any():
        raise ValueError('Patient manifest contains duplicate filenames')
    metadata_columns = ['filename', 'patient_id']
    if 'slide_id' in source:
        metadata_columns.append('slide_id')
    inventory = inventory.merge(
        source[metadata_columns], on='filename', how='left', validate='one_to_one'
    )
    if inventory['patient_id'].isna().any():
        missing_count = int(inventory['patient_id'].isna().sum())
        raise ValueError(f'Patient manifest did not match {missing_count} tiles')
    inference_level = 'patient_cluster_confirmatory'
    print('Patients:', inventory['patient_id'].nunique())
    if 'slide_id' in inventory:
        print('Slides:', inventory['slide_id'].nunique())
else:
    inventory['patient_id'] = pd.NA
    inventory['slide_id'] = pd.NA
    inference_level = 'tile_level_exploratory'
    print(
        'No official tile-to-patient manifest was found. Confirmatory patient-level '
        'bootstrap and patient-stratified cohort claims are disabled.'
    )

inventory.to_csv(OUTPUT_DIR / 'crc_val_inventory_with_metadata.csv', index=False)


## Frozen label protocols

The native replication retains all nine CRC classes. The secondary transfer
excludes CRC smooth muscle and merges CRC debris and mucus into the single Kather
debris/mucus target. Kather complex stroma is excluded from secondary training;
it is not relabeled as smooth muscle.


In [ ]:
native_labels = pd.DataFrame(
    {
        'crc_class': config['primary_native_replication']['classes'],
        'native_label': range(len(config['primary_native_replication']['classes'])),
    }
)
coarse_map = config['secondary_coarse_transfer']['external_to_common']
coarse_labels = pd.DataFrame(
    [{'crc_class': key, 'common_class': value} for key, value in coarse_map.items()]
).sort_values(['common_class', 'crc_class'])
coarse_class_order = sorted(coarse_labels['common_class'].unique())
coarse_class_to_label = {name: index for index, name in enumerate(coarse_class_order)}
coarse_labels['common_label'] = coarse_labels['common_class'].map(coarse_class_to_label)

native_labels.to_csv(OUTPUT_DIR / 'native_nine_class_mapping.csv', index=False)
coarse_labels.to_csv(OUTPUT_DIR / 'kather_crc_common_seven_mapping.csv', index=False)
display(native_labels)
display(coarse_labels)


## Prediction-independent attribution cohorts

Sampling uses labels, official patient IDs when available, and the fixed seed
2027. It never uses predictions, confidence, attribution maps, or model identity.
When patient IDs are available, each class is sampled round-robin across patients.
Without them, the generated cohorts are explicitly exploratory class-stratified
tile samples.


In [ ]:
def balanced_sample(frame, class_column, per_class, seed):
    rng = np.random.default_rng(seed)
    pieces = []
    for class_name, class_frame in frame.groupby(class_column, sort=True):
        class_frame = class_frame.copy()
        if len(class_frame) < per_class:
            raise ValueError(f'{class_name} has fewer than {per_class} images')
        if patient_metadata_available:
            patient_order = list(class_frame['patient_id'].drop_duplicates())
            rng.shuffle(patient_order)
            patient_frames = {
                patient: class_frame[class_frame['patient_id'].eq(patient)].sample(
                    frac=1, random_state=int(rng.integers(0, 2**31 - 1))
                )
                for patient in patient_order
            }
            offsets = {patient: 0 for patient in patient_order}
            selected = []
            while len(selected) < per_class:
                advanced = False
                for patient in patient_order:
                    offset = offsets[patient]
                    patient_frame = patient_frames[patient]
                    if offset < len(patient_frame):
                        selected.append(patient_frame.iloc[offset])
                        offsets[patient] += 1
                        advanced = True
                        if len(selected) == per_class:
                            break
                if not advanced:
                    raise RuntimeError(f'Could not complete class {class_name}')
            pieces.append(pd.DataFrame(selected))
        else:
            pieces.append(
                class_frame.sample(
                    n=per_class,
                    replace=False,
                    random_state=int(rng.integers(0, 2**31 - 1)),
                )
            )
    cohort = pd.concat(pieces, ignore_index=True)
    cohort.insert(
        0,
        'cohort_id',
        cohort['relative_path'].map(
            lambda value: hashlib.sha256(value.encode('utf-8')).hexdigest()[:16]
        ),
    )
    cohort['selection_seed'] = int(seed)
    cohort['selected_using_predictions'] = False
    cohort['selected_using_attributions'] = False
    cohort['patient_stratified'] = bool(patient_metadata_available)
    cohort['inference_level'] = inference_level
    return cohort.sort_values([class_column, 'relative_path']).reset_index(drop=True)


native_manifest = inventory.merge(native_labels, on='crc_class', validate='many_to_one')
native_cohort = balanced_sample(
    native_manifest,
    'crc_class',
    config['primary_native_replication']['images_per_attribution_class'],
    config['cohort_selection_seed'],
)

coarse_manifest = inventory[inventory['crc_class'].isin(coarse_map)].copy()
coarse_manifest['common_class'] = coarse_manifest['crc_class'].map(coarse_map)
coarse_manifest['common_label'] = coarse_manifest['common_class'].map(coarse_class_to_label)
coarse_cohort = balanced_sample(
    coarse_manifest,
    'common_class',
    config['secondary_coarse_transfer']['images_per_attribution_class'],
    config['cohort_selection_seed'],
)


def freeze_manifest(frame, path):
    if path.is_file():
        existing = pd.read_csv(path)
        if existing['cohort_id'].tolist() != frame['cohort_id'].tolist():
            raise RuntimeError(f'Frozen cohort differs from proposed cohort: {path}')
    else:
        frame.to_csv(path, index=False)
    return hashlib.sha256(path.read_bytes()).hexdigest()


cohort_suffix = (
    'patient_stratified' if patient_metadata_available else 'tile_exploratory'
)
native_path = OUTPUT_DIR / f'native_nine_class_attribution_cohort_{cohort_suffix}.csv'
coarse_path = OUTPUT_DIR / f'common_seven_attribution_cohort_{cohort_suffix}.csv'
native_hash = freeze_manifest(native_cohort, native_path)
coarse_hash = freeze_manifest(coarse_cohort, coarse_path)

display(native_cohort.groupby('crc_class').size().rename('images'))
display(coarse_cohort.groupby('common_class').size().rename('images'))
print('Native cohort SHA-256:', native_hash)
print('Coarse cohort SHA-256:', coarse_hash)


## Readiness record

`ready_for_confirmatory_patient_inference` is intentionally false until an
official, complete patient manifest is joined. Tile-level exploratory work may
proceed, but it must not be presented as patient-cluster inference.


In [ ]:
readiness = {
    'protocol_version': config['protocol_version'],
    'dataset_images': int(len(inventory)),
    'dataset_classes': int(inventory['crc_class'].nunique()),
    'inventory_sha256': inventory_hash,
    'patient_metadata_available': bool(patient_metadata_available),
    'inference_level': inference_level,
    'ready_for_confirmatory_patient_inference': bool(patient_metadata_available),
    'native_cohort_images': int(len(native_cohort)),
    'native_cohort_sha256': native_hash,
    'coarse_cohort_images': int(len(coarse_cohort)),
    'coarse_cohort_sha256': coarse_hash,
    'external_predictions_examined': False,
}
(OUTPUT_DIR / 'preflight_readiness.json').write_text(
    json.dumps(readiness, indent=2), encoding='utf-8'
)
display(pd.Series(readiness, name='value').to_frame())


## Locked downstream execution

After the preflight is archived:

1. train the native nine-class ResNet18, DINOv2, and UNI pipelines on
   NCT-CRC-HE-100K with seeds 11, 89, and 181;
2. evaluate all 7,180 CRC-VAL tiles once and save image x seed logits;
3. separately train common-seven Kather models and evaluate the 6,588 eligible
   CRC tiles (all classes except MUS, with DEB and MUC merged);
4. run only Grad-CAM or gradient-weighted rollout on the frozen cohort, common
   14 x 14 grid, normalized-zero perturbation, and five random deletion repeats;
5. average seeds within images before patient-cluster paired inference;
6. report attribution-occlusion Spearman, relative target-logit top-minus-random
   deletion AUC, and the proportion of images for which top deletion beats random.

Do not compare raw cross-model logit magnitudes. Do not reopen every Kather
perturbation/grid analysis unless the external ranking materially conflicts with
the Kather primary result. Highlighted patches indicate contribution to model
predictions, not biological causation.
